
<div align="center">

<img src="https://capsule-render.vercel.app/api?type=waving&color=0:050B18,50:0B3D91,100:123B60&height=190&section=header&text=Luis%20Barahona&fontSize=42&fontColor=FFFFFF&fontAlignY=38&desc=DATA%20ENGINEERING%20%7C%20ARTIFICIAL%20INTELLIGENCE%20%7C%20DEVELOPER&descAlignY=62&descSize=14" width="50%" alt="Luis Barahona - Data Engineering and AI">

<img src="https://readme-typing-svg.demolab.com?font=Fira+Code&weight=500&size=17&duration=3500&pause=1200&color=38BDF8&center=true&vCenter=true&width=600&lines=Python+%7C+SQL+%7C+Data+Engineering;Artificial+Intelligence+%7C+Machine+Learning;Data+Science+%7C+Cloud+Computing;Building+Technology+for+the+Future" alt="Technical expertise">

</div>

---
<div align="center">
### Professional Profile

I am a Mexican Data Engineer and a 2026 graduate of Universidad del Caribe, passionate about technology, innovation, and continuous learning.
Beyond technical expertise, I value communication, teamwork, collaboration, and building meaningful connections with others. I believe that every person offers an opportunity to learn something new, and I am always motivated to share knowledge, explore new ideas, and grow both personally and professionally.

---

<div align="center">

<img src="https://skillicons.dev/icons?i=python,mysql,postgres,git,github,react,firebase,linux,docker,vscode&theme=dark" alt="Technical skills">


<strong>Data Engineering ·  Data Analytics · Data science · Artificial Intelligence · Machine Learning · Data Analytics · Developer</strong>

</div>

---
<div align="center">

| Platform | Contact |
|:---|:---|
| Email | [190300395@ucaribe.edu.mx](mailto:190300395@ucaribe.edu.mx) |
| GitHub | [github.com/LuisBarahonaMX](https://github.com/LuisBarahonaMX) |
| LinkedIn | [linkedin.com/in/luis-barahona-mx](https://www.linkedin.com/in/luis-barahona-mx/) |
| Instagram | [@luisbarahonamx](https://www.instagram.com/luisbarahonamx/) |
| X | [@luisbarahonaMX](https://x.com/luisbarahonaMX) |
| Website | [SIKO AI Platform](https://sikoai.web.app/) |
</div>



#Multimedia downloader 2.0 — YouTube , facebook, instagram, x, , ticktock
A Python Colab notebook that downloads videos from various platforms, such as YouTube, Facebook, Instagram, X, LinkedIn, among others.


**Usage:** Execute the cells in order. Download only content that you own or for which you have authorization. Some platforms restrict downloads, and videos protected by DRM are not supported. Files in `/content/descargas` are temporary and may be lost when the Colab session ends.

In [1]:
# 1. Instalar dependencias en Google Colab
import sys, subprocess, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp", "ipywidgets"], check=True)
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
import yt_dlp
print("yt-dlp:", yt_dlp.version.__version__)
print("FFmpeg:", shutil.which("ffmpeg"))


yt-dlp: 2026.10.26
FFmpeg: /usr/bin/ffmpeg


In [3]:
# 2. Interfaz de descargas
import ipywidgets as w
from IPython.display import display, clear_output, FileLink
from pathlib import Path
from urllib.parse import urlparse
import subprocess, sys, re, time, os, zipfile, tempfile

DESTINO = Path("/content/descargas") if Path("/content").exists() else Path.cwd() / "descargas"
DESTINO.mkdir(parents=True, exist_ok=True)

url = w.Text(description="Enlace:", placeholder="https://www.youtube.com/watch?v=...", layout=w.Layout(width="95%"))
tipo = w.ToggleButtons(options=[("Audio MP3", "mp3"), ("Video MP4", "mp4")], description="Formato:")
calidad = w.Dropdown(options=[("Mejor disponible", "best"), ("1080p", "1080"), ("720p", "720"), ("480p", "480"), ("360p", "360")], description="Video:")
lista = w.Checkbox(value=False, description="Permitir lista de reproducción (puede descargar muchos archivos)")
cookies = w.FileUpload(accept=".txt", multiple=False, description="Cookies (opcional)")
descargar = w.Button(description="Iniciar descarga", button_style="success", icon="download")
archivos = w.Button(description="Mostrar archivos", icon="folder-open")
salida = w.Output()

def validar_enlace(valor):
    try:
        p = urlparse(valor.strip())
        return p.scheme in ("http", "https") and bool(p.hostname) and not p.username and not p.password
    except ValueError:
        return False

def contenido_upload(widget):
    # Compatible con ipywidgets 7 (dict) y 8 (tupla).
    v = widget.value
    if not v:
        return None
    item = next(iter(v.values())) if isinstance(v, dict) else v[0]
    contenido = item["content"]
    return bytes(contenido)

def mostrar_archivos(_=None):
    with salida:
        print("\nArchivos disponibles:")
        encontrados = [p for p in sorted(DESTINO.iterdir()) if p.is_file() and not p.name.endswith((".part", ".ytdl", ".temp"))]
        if not encontrados:
            print("Todavía no hay archivos descargados.")
        for p in encontrados[-30:]:
            print(f"• {p.name} ({p.stat().st_size / 1024**2:.1f} MB)")
            display(FileLink(str(p)))
        print("Para bajar archivos al equipo en Colab, utiliza los enlaces o la celda 3.")

def ejecutar(_):
    descargar.disabled = True
    try:
        with salida:
            clear_output(wait=True)
            enlace = url.value.strip()
            if not validar_enlace(enlace):
                print("Error: introduce una URL HTTP o HTTPS válida.")
                return
            if not lista.value and ("list=" in enlace or "playlist" in enlace.lower()):
                print("Modo seguro: solo se descargará un video (--no-playlist).")
            seleccion = calidad.value
            cmd = [
                sys.executable, "-m", "yt_dlp", "--ignore-config",
                "--newline", "--no-overwrites",
                "--retries", "3", "--fragment-retries", "3",
                "--socket-timeout", "30",
                "--restrict-filenames",
                "-P", str(DESTINO),
                "-o", "%(title).100s_%(id)s.%(ext)s",
            ]
            cmd.append("--yes-playlist" if lista.value else "--no-playlist")
            if tipo.value == "mp3":
                cmd += ["-f", "bestaudio/best", "-x", "--audio-format", "mp3", "--audio-quality", "0"]
            else:
                limite = "" if seleccion == "best" else f"[height<={seleccion}]"
                # Prefiere MP4 y M4A, con alternativas si no existen.
                selector = f"bv*[ext=mp4]{limite}+ba[ext=m4a]/bv*{limite}+ba/b{limite}/b"
                cmd += ["-f", selector, "--merge-output-format", "mp4", "--recode-video", "mp4"]
            ruta_cookies = None
            datos = contenido_upload(cookies)
            if datos:
                fd, ruta_cookies = tempfile.mkstemp(prefix="ytcookies_", suffix=".txt")
                with os.fdopen(fd, "wb") as f:
                    f.write(datos)
                os.chmod(ruta_cookies, 0o600)
                cmd += ["--cookies", ruta_cookies]
                print("Cookies temporales habilitadas. Úsalas solo en una sesión privada.")
            cmd.append(enlace)
            print(f"Iniciando descarga: {tipo.value.upper()}; calidad: {seleccion}; lista: {'sí' if lista.value else 'no'}")
            print("Tiempo máximo por ejecución: 30 minutos. Reintentos limitados.\n")
            try:
                resultado = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                           text=True, errors="replace", timeout=1800, check=False)
                lineas = resultado.stdout.splitlines()
                # Mantener el registro manejable: progreso final y mensajes de error.
                progreso = [x for x in lineas if "[download]" in x and "%" in x]
                mensajes = [x for x in lineas if any(t in x.lower() for t in ("error:", "warning:", "destination:", "merging formats", "extractaudio", "deleting original"))]
                for linea in (progreso[-5:] + mensajes[-20:]):
                    print(linea)
                print("\n" + ("Descarga terminada." if resultado.returncode == 0 else f"Falló la descarga (código {resultado.returncode})."))
                if resultado.returncode != 0 and not mensajes:
                    print("\n".join(lineas[-12:]))
            except subprocess.TimeoutExpired:
                print("Se alcanzó el límite de 30 minutos; el proceso se detuvo.")
            finally:
                if ruta_cookies:
                    try:
                        os.unlink(ruta_cookies)
                    except OSError:
                        pass
            mostrar_archivos()
    except Exception as e:
        with salida:
            print(f"Error inesperado: {type(e).__name__}: {e}")
    finally:
        descargar.disabled = False

descargar.on_click(ejecutar)
archivos.on_click(mostrar_archivos)
display(w.VBox([
    w.HTML("<h3>Descargador multimedia 2.0</h3>"),
    url, w.HBox([tipo, calidad]), lista,
    w.HTML("<small>Opcional: cookies.txt. Contiene credenciales sensibles; evita usarlo en notebooks compartidos.</small>"),
    cookies, w.HBox([descargar, archivos]), salida
]))


In [ ]:
# 3. Descargar resultados al equipo (opcional; ejecutar después de terminar)
# En Colab, genera un ZIP con los archivos multimedia y abre la descarga del navegador.
from pathlib import Path
import zipfile
from IPython.display import display, FileLink

media = [p for p in DESTINO.iterdir() if p.is_file() and p.suffix.lower() in (".mp3", ".mp4", ".m4a", ".webm", ".mkv")]
if not media:
    print("No hay archivos multimedia disponibles.")
else:
    zip_path = DESTINO.parent / "descargas_multimedia.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as z:
        for p in media:
            z.write(p, arcname=p.name)
    print(f"ZIP creado: {zip_path.name} ({zip_path.stat().st_size / 1024**2:.1f} MB)")
    display(FileLink(str(zip_path)))
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print("Fuera de Colab: utiliza el enlace mostrado arriba.")
